# Score-based search (XGES / hill climbing / Tabu search / exact search / GIES)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/score_search.ipynb)

XGES, hill climbing, and Tabu search look for the graph with the best BIC score. XGES searches
over equivalence classes, as GES does, with deletions first. Hill climbing adds, removes, or
reverses one edge of a DAG at a time and stops when no move improves the score; Tabu search keeps
going past that point and returns the best DAG it visits. All three return a CPDAG and suit
linear-Gaussian data. Exact search finds the best-scoring DAG on a few variables, and GIES also
learns from interventions.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.12.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.12.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 2,000 samples.

In [3]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

2000 observations, 10 variables, seed 7


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 332" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="332" fill="#fafbfc"/><line x1="312.82" y1="44.82" x2="242.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,39.30 310.06,40.65 315.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="144.93" y1="163.00" x2="409.13" y2="134.59" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.07,133.52 408.60,129.62 409.67,139.56" fill="#3a4658" stroke-linejoin="round"/><line x1="113.24" y1="177.70" x2="42.60" y2="227.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,171.91 110.35,173.62 116.14,181.78" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="151.70" x2="42.84" y2="105.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,157.22 115.57,147.53 110.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="128.29" x2="244.82" y2="100.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,129.96 410.16,123.36 408.49,133.22" fill="#3a4658" stroke-linejoin="round"/><line x1="410.18" y1="124.86" x2="144.33" y2="36.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,128.02 411.76,120.11 408.60,129.60" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="139.19" x2="344.37" y2="160.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,136.12 408.58,134.43 411.65,143.95" fill="#3a4658" stroke-linejoin="round"/><line x1="139.66" y1="224.85" x2="218.45" y2="116.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="224.34,108.76 214.41,113.89 222.49,119.79" fill="#3a4658" stroke-linejoin="round"/><line x1="142.92" y1="242.40" x2="212.67" y2="287.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.08,292.82 215.38,283.20 209.96,291.60" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="235.35" x2="45.00" y2="235.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,235.35 109.00,230.35 109.00,240.35" fill="#3a4658" stroke-linejoin="round"/><line x1="212.82" y1="85.57" x2="142.84" y2="39.30" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.16,91.09 215.57,81.40 210.06,89.74" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="105.43" x2="312.82" y2="151.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,157.22 315.57,147.53 310.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="98.26" x2="45.00" y2="98.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,98.26 209.00,93.26 209.00,103.26" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="44.82" x2="42.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,39.30 110.06,40.65 115.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="292.81" x2="44.37" y2="239.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,295.88 211.65,288.05 208.58,297.57" fill="#3a4658" stroke-linejoin="round"/><circle cx="332.00" cy="32.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="32.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="132.00" cy="164.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="164.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="132.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="132.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominan

## Fit XGES, hill climbing, and Tabu search

`metadata["score"]` is the BIC score of the returned graph, which each search minimizes. A larger
`penalty_discount` weighs the BIC penalty more and gives a sparser graph. With
`extended_search=False`, XGES skips its extended search for a faster fit, and `tabu_length` sets
how many recently visited DAGs Tabu search may not return to.

In [4]:
results = {
    "XGES": andrey.xges(X),
    "Hill climbing": andrey.hc(X),
    "Tabu search": andrey.tabu(X),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}")
    display(andrey.viz.draw(result.structure))

XGES: cpdag, score -22359.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

Hill climbing: cpdag, score -22340.4


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 564 319" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="564" height="319" fill="#fafbfc"/><line x1="45.00" y1="230.47" x2="119.00" y2="230.47" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="441.27" y1="163.78" x2="515.60" y2="236.92" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="522.73,243.93 519.11,233.35 512.10,240.48" fill="#3a4658" stroke-linejoin="round"/><line x1="410.71" y1="145.96" x2="144.03" y2="37.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.97,149.75 412.60,141.34 408.82,150.59" fill="#3a4658" stroke-linejoin="round"/><line x1="409.00" y1="154.66" x2="345.00" y2="154.66" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,154.66 409.00,149.66 409.00,159.66" fill="#3a4658" stroke-linejoin="round"/><line x1="509.04" y1="251.75" x2="144.98" y2="231.20" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.02,252.31 509.32,246.76 508.75,256.74" fill="#3a4658" stroke-linejoin="round"/><line x1="509.94" y1="246.52" x2="244.47" y2="168.03" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.53,249.36 511.36,241.73 508.53,251.32" fill="#3a4658" stroke-linejoin="round"/><line x1="509.32" y1="256.89" x2="344.82" y2="284.75" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.18,255.22 508.49,251.96 510.16,261.82" fill="#3a4658" stroke-linejoin="round"/><line x1="41.19" y1="139.66" x2="122.81" y2="221.27" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="44.82" y1="132.64" x2="219.18" y2="162.17" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="41.27" y1="121.35" x2="115.60" y2="48.21" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.73,41.20 112.10,44.65 119.11,51.78" fill="#3a4658" stroke-linejoin="round"/><line x1="44.96" y1="131.52" x2="319.04" y2="153.62" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="142.84" y1="223.29" x2="221.16" y2="171.51" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="144.51" y1="234.00" x2="319.49" y2="283.39" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="144.16" y1="225.86" x2="319.84" y2="159.27" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="244.94" y1="163.09" x2="319.06" y2="155.92" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="45.00" y1="32.08" x2="109.00" y2="32.08" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,32.08 109.00,27.08 109.00,37.08" fill="#3a4658" stroke-linejoin="round"/><circle cx="32.00" cy="230.47" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="230.47" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="432.00" cy="154.66" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="154.66" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="532.00" cy="253.05" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="253.05" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="130.47" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="130.47" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="132.00" cy="230.47" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="230.47" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="232.00" cy="164.34" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><

Tabu search: cpdag, score -22359.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

## Compare with GES over ten datasets

Structural Hamming distance (SHD) counts errors against the truth's CPDAG; lower is better. The
empty graph's SHD is the number of true edges, so an estimate at or above it has not improved on
returning no edges. One dataset can favor any method, so the table covers ten seeds and their
median.

In [5]:
methods = {
    "GES": lambda X: andrey.ges(X),
    "XGES": lambda X: andrey.xges(X),
    "Hill climbing": lambda X: andrey.hc(X),
    "Tabu search": lambda X: andrey.tabu(X),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="cpdag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

seed,0,1,2,3,4,5,6,7,8,9,median
Empty graph,15,15,15,15,15,15,15,15,15,15,15.0
GES,21,10,0,10,17,7,11,17,0,0,10.0
XGES,6,8,0,1,1,0,0,6,0,0,0.5
Hill climbing,23,17,0,13,17,21,9,10,7,8,11.5
Tabu search,22,0,0,9,15,9,0,6,0,0,3.0


Tabu search starts with hill climbing's moves, so its score is never worse than hill
climbing's on the same data. The [0.3.0 release card](https://andrey.abel.ai/docs/benchmarks/0.3.0.html)
compares XGES with the xges package and the other two with bnlearn, on larger graphs with ten rows
per variable.

## Hybrid search: MMHC and RSMAX2

MMHC learns a skeleton with conditional-independence tests, then runs hill climbing that joins only
the skeleton's pairs. `rsmax2` generalizes it: `restrict` picks the test-based phase (`"mmpc"`,
`"iamb"`, `"inter_iamb"`, or `"pc"`) and `maximize` the score search (`"hc"` or `"tabu"`).

In [6]:
hybrids = {
    "MMHC": andrey.mmhc(X),
    "RSMAX2 (IAMB restrict)": andrey.rsmax2(X, restrict="iamb"),
}
for name, result in hybrids.items():
    shd = metrics.score(result, truth)["shd"]
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}, SHD {shd}")

MMHC: cpdag, score -20302.8, SHD 9
RSMAX2 (IAMB restrict): cpdag, score -20079.9, SHD 10


## Exact search

`exact_search` scores every parent set of every variable and returns the CPDAG of the DAGs
with the best BIC score; `metadata["dag"]` keeps the one it found. Its run time at least
doubles with each added variable, so keep it to about 20 variables.

In [7]:
exact = andrey.exact_search(X)
print("Exact search:", exact.structure.kind)
display(andrey.viz.draw(exact.structure))

Exact search: cpdag


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

## Interventions: GIES

`gies` also takes the rows' interventions: `True` where a row's value of a variable was set from
outside. A chain from observational data stays undirected; here the last 1,000 rows set `x1`, which
orients both of its edges. The columns come in an order drawn from the seed, and the interventions
follow them.

In [8]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges, cause first


def sample_intervened(
    seed: int, n: int = 3000, shuffle: bool = True
) -> tuple[pd.DataFrame, np.ndarray]:
    """Draw n rows of the chain, the last 1,000 setting x1 from outside, and the interventions."""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = 0.8 * x0 + rng.standard_normal(n)
    x1[n - 1000 :] = rng.standard_normal(1000)  # the last 1,000 rows set x1 from outside
    x2 = 0.8 * x1 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2})
    frame = frame[rng.permutation(frame.columns)] if shuffle else frame
    intervened = np.zeros((n, 3), dtype=bool)
    intervened[n - 1000 :, frame.columns.get_loc("x1")] = True
    return frame, intervened


gies_out = andrey.gies(*sample_intervened(seed=1))
print(gies_out.structure.oriented_edges())
display(andrey.viz.draw(gies_out.structure))

[('x1', 'x2', 'directed'), ('x0', 'x1', 'directed')]


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="193.00" y1="120.00" x2="257.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,120.00 257.00,115.00 257.00,125.00" fill="#3a4658" stroke-linejoin="round"/><line x1="157.00" y1="120.00" x2="93.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.00,120.00 157.00,115.00 157.00,125.00" fill="#3a4658" stroke-linejoin="round"/><circle cx="180.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="280.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text><circle cx="80.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text></svg>'

## Use another dataset

Change `seed`, or the `d` and `n` passed to `benchmarks.scm` and `sample`, and rerun.